# Projet 1: Prédiction du comportement des clients avec l'IA dans une banque

Ce notebook contient l'analyse exploratoire des données (EDA), l'analyse statistique descriptive, la visualisation des caractéristiques, ainsi que l'entraînement et l'évaluation d'un modèle d'ensemble de classification (Régression Logistique, Forêt Aléatoire, et un perceptron multicouche PyTorch MLP) pour prédire si un client souscrira à un dépôt à terme (campagne marketing).

In [8]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

sns.set_theme(style="darkgrid")
print("Librairies chargées avec succès.")

ModuleNotFoundError: No module named 'pandas'

## 1. Chargement et Analyse Descriptive des Données

In [ ]:
# Charger le dataset
df = pd.read_csv("dataset/bank.csv", sep=";")
print(f"Dimensions du dataset: {df.shape[0]} lignes, {df.shape[1]} colonnes.")
df.head()

In [ ]:
# Information générale
df.info()

In [ ]:
# Description statistique des variables numériques
df.describe()

In [ ]:
# Distribution de la variable cible (y)
class_counts = df['y'].value_counts()
class_pct = df['y'].value_counts(normalize=True) * 100
print("Distribution des classes (Souscription) :")
for idx, val in class_counts.items():
    print(f"  {idx}: {val} ({class_pct[idx]:.2f}%)")

## 2. Visualisation Graphique (EDA)

In [ ]:
# Visualisation de la distribution de l'âge des clients
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='age', hue='y', kde=True, bins=30, palette='viridis', multiple='stack')
plt.title("Distribution de l'âge des clients selon la souscription (y)")
plt.xlabel("Âge")
plt.ylabel("Nombre de clients")
plt.show()

In [ ]:
# Visualisation du solde bancaire annuel moyen selon la souscription
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='y', y='balance', palette='pastel')
plt.title("Solde bancaire annuel moyen vs Dépôt à terme")
plt.xlabel("Souscription (y)")
plt.ylabel("Solde (balance)")
plt.yscale('log') # Utilisation de l'échelle logarithmique pour les valeurs extrêmes
plt.show()

In [ ]:
# Impact des emplois (job) sur la souscription
plt.figure(figsize=(14, 6))
sns.countplot(data=df, x='job', hue='y', palette='Set2')
plt.title("Souscription au dépôt à terme selon le type d'emploi")
plt.xticks(rotation=45)
plt.xlabel("Emploi")
plt.ylabel("Nombre")
plt.legend(title="Souscription")
plt.tight_layout()
plt.show()

## 3. Prétraitement des Données et Corrélation

In [ ]:
# Mapper la cible 'y' en binaire
df['y_numeric'] = df['y'].map({'yes': 1, 'no': 0})

# Variables numériques pour la matrice de corrélation
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Matrice de corrélation des variables numériques")
plt.show()

## 4. Entraînement et Évaluation des Modèles

In [ ]:
X = df.drop(columns=['y', 'y_numeric'])
y = df['y_numeric']

num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"X_train après preprocessing: {X_train_proc.shape}")

In [ ]:
# Modèle 1: Régression Logistique
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_proc, y_train)
lr_probs = lr.predict_proba(X_test_proc)[:, 1]
lr_preds = lr.predict(X_test_proc)

# Modèle 2: Forêt Aléatoire
rf = RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42)
rf.fit(X_train_proc, y_train)
rf_probs = rf.predict_proba(X_test_proc)[:, 1]
rf_preds = rf.predict(X_test_proc)

print("Modèles de base entraînés.")

In [ ]:
# Visualisation des Courbes ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Régression Logistique (AUC = {roc_auc_score(y_test, lr_probs):.4f})')
plt.plot(fpr_rf, tpr_rf, label=f'Forêt Aléatoire (AUC = {roc_auc_score(y_test, rf_probs):.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Aléatoire (AUC = 0.5000)')
plt.title("Courbes ROC")
plt.xlabel("Taux de Faux Positifs")
plt.ylabel("Taux de Vrais Positifs")
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Matrice de confusion pour la Forêt Aléatoire
cm = confusion_matrix(y_test, rf_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Non', 'Oui'], yticklabels=['Non', 'Oui'])
plt.title("Matrice de Confusion - Forêt Aléatoire")
plt.xlabel("Prédiction")
plt.ylabel("Vérité Terrain")
plt.show()